# RLSF: the two pre-registered arms

---
## 1 — Setup

In [ ]:
# 7B bf16 policy, a frozen reference adapter and the COMET encoder share the card, and GRPO
# holds logits for a micro-batch over a 151,936-token vocabulary: 40 GB, not 24.
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from pathlib import Path

if not Path('manage.py').exists():
    if not Path('Style-Aware-MT/manage.py').exists():
        !git clone --branch feat/rlsf-implementation https://github.com/prnamhr/Style-Aware-MT.git
    %cd Style-Aware-MT
!git pull --ff-only
!git rev-parse --short HEAD

In [ ]:
import subprocess
import sys

PY = sys.executable
COMET_PY = '.venv-comet/bin/python'
print('kernel', PY)

In [ ]:
# %pip installs into the kernel; !pip may not.
%pip install -q -r requirements.txt

In [ ]:
import numpy
import transformers

# COMET gets its own interpreter. Installing requirements-comet.txt into the kernel downgrades
# transformers and numpy under the generator, which then runs on a stack nothing else uses.
if not Path(COMET_PY).exists():
    pip = [COMET_PY, '-m', 'pip', 'install', '-q']
    subprocess.run([PY, '-m', 'venv', '.venv-comet'], check=True)
    subprocess.run([*pip, '--upgrade', 'pip'], check=True)
    subprocess.run([*pip, 'setuptools<81'], check=True)
    subprocess.run([*pip, '-r', 'requirements-comet.txt'], check=True)
subprocess.run([COMET_PY, '-c', 'import comet; print("comet ok")'], check=True)

print(f'kernel transformers {transformers.__version__}, numpy {numpy.__version__}')
assert transformers.__version__.startswith('5.'), "COMET's pins landed in the kernel"
assert numpy.__version__.startswith('2.'), "COMET's pins landed in the kernel"

In [ ]:
import getpass
import logging
import os

# Read once, when the allocator initializes: at 31.6 of 32 GB it is fragmentation,
# not capacity, that fails first. src.rlsf.train sets the same default.
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

for var in ('OPENAI_API_KEY', 'HF_TOKEN'):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f'{var}: ')
logging.getLogger('httpx').setLevel(logging.WARNING)
print({var: bool(os.environ.get(var)) for var in ('OPENAI_API_KEY', 'HF_TOKEN')})

In [ ]:
check = '''
import os
from huggingface_hub import HfApi, __version__
api, tok = HfApi(), os.environ.get("HF_TOKEN")
print("hub", __version__, "| whoami:", api.whoami(token=tok)["name"])
api.list_repo_files("Unbabel/wmt22-cometkiwi-da", token=tok)
print("cometkiwi access OK")
'''
subprocess.run([COMET_PY, '-c', check], check=True)

---
## 2 — Pre-flight

Everything the pre-registration fixes, checked before a GPU-hour is spent on either arm.

In [ ]:
import hashlib
import json
import math

import yaml

from src.rlsf.config import (
    drift_rule,
    judge_concurrency,
    load_config,
    make_judge_client,
    rollout_batch,
)
from src.rlsf.reward import load_train_template
from src.rlsf.train import arm_path, arm_reward_config, sidecar

CONFIG = 'configs/rlsf.yaml'
cfg = load_config(CONFIG)

# The arms, in the order they run. Named here once; every cell below reads this map.
ARMS = {'RL-Metric': 'w3_0.0', 'RLSF-Judge': 'w3_2.0'}
STEPS = 500                      # shared: the arms are not comparable at different lengths
G = cfg['rlsf']['rollout']['group_size']

print(f"{'arm':12s} {'cell':8s} {'bleu':>8s} {'kiwi':>8s} {'judge':>8s} {'||w||':>7s} "
      f"{'judge var share':>16s}")
for name, cell in ARMS.items():
    rc = arm_reward_config(cfg, cell)
    w = rc.weights
    print(f"{name:12s} {cell:8s} {w['bleu']:8.4f} {w['kiwi']:8.4f} {w['judge']:8.4f} "
          f"{math.hypot(*w.values()):7.4f} {rc.w_judge ** 2:16.4f}")

In [ ]:
# -- the arms are the two the pre-registration names, at the omega it tabulates
PREREG = {'w3_0.0': (0.7071, 0.7071, 0.0), 'w3_2.0': (0.4082, 0.4082, 0.8165)}
for cell, want in PREREG.items():
    rc = arm_reward_config(cfg, cell)
    got = (rc.w_bleu, rc.w_kiwi, rc.w_judge)
    assert all(math.isclose(a, b, abs_tol=5e-5) for a, b in zip(got, want)), (cell, got, want)
    # Unit norm in both arms, or omega doubles as a step size and lr means two things.
    assert math.isclose(math.hypot(*got), 1.0, abs_tol=1e-9)

# -- held identical across the arms: the contrast is omega and nothing else
assert cfg['rlsf']['reference']['beta'] == 0.05
assert cfg['rlsf']['seed'] == 42
assert cfg['rlsf']['train']['learning_rate'] == 1e-6
assert cfg['rlsf']['train']['num_iterations'] == 4
assert cfg['rlsf']['train']['epsilon'] == 0.2
assert cfg['rlsf']['train']['loss_type'] == 'dapo'
assert (cfg['rlsf']['rollout']['temperature'], cfg['rlsf']['rollout']['top_p']) == (1.0, 0.95)
assert cfg['rlsf']['rollout']['prompts_per_step'] == 16 and G == 4
for cell in ARMS.values():
    rc = arm_reward_config(cfg, cell)
    assert (rc.len_min_ratio, rc.len_max_ratio) == (0.5, 2.0)
    assert rc.on_violation == 'floor' and rc.normalization == 'group_z'
print('omega and the held-constant settings match the pre-registration')

In [ ]:
gen = cfg['generator']
assert gen['model'] == 'Qwen/Qwen2.5-7B-Instruct', gen['model']
assert gen['load_in_4bit'] is False, 'quantizing redefines the frozen base'
assert gen['adapter_path'] == 'models/peft_lora_r32_lr2e-4/checkpoint-1358'
assert cfg['rlsf']['reference']['adapter_path'] == gen['adapter_path'], (
    'the KL reference must be a frozen copy of the initialization'
)
assert Path(gen['adapter_path'], 'adapter_config.json').exists(), (
    f"{gen['adapter_path']} is missing; copy the frozen PEFT checkpoint onto this host first"
)
print(f"policy {gen['model']} + {gen['adapter_path']}, reference frozen at the same checkpoint")

In [ ]:
rule = drift_rule(cfg)
assert vars(rule) == {
    'feature': 'marker_rate', 'baseline_steps': 20, 'window': 5,
    'k_sigma': 4.0, 'min_delta': 0.23,
}, vars(rule)
assert STEPS > rule.first_testable_step, (
    f'{STEPS} steps cannot test a rule that needs {rule.first_testable_step}'
)
print(f"drift rule: {rule.feature}, baseline {rule.baseline_steps} steps, window "
      f"{rule.window}, band max({rule.min_delta}, {rule.k_sigma} se)")
print(f"first step at which it can fire: {rule.first_testable_step}")

In [ ]:
raters = {yaml.safe_load(Path(p).read_text())['judge']['model']
          for p in ('configs/judge_eval.yaml', 'configs/judge_eval_gpt.yaml')}
assert cfg['judge']['model'] not in raters, (cfg['judge']['model'], raters)
assert cfg['judge']['temperature'] == 0.0 and cfg['judge']['seed'] == 42

# -- the rubric must be the frozen training one; load_train_template raises on drift
text = load_train_template()
digest = hashlib.sha256(text.encode()).hexdigest()
frozen = json.loads(Path('prompts/hashes.json').read_text())['templates']
assert digest == frozen['judge_train.txt']['sha256']
assert cfg['template_file'] == 'prompts/judge_train.txt', 'the eval rubric would be circular'
print(f"reward judge {cfg['judge']['model']}, distinct from {sorted(raters)}")
print(f"rubric verified {digest[:16]}")

In [ ]:
# -- the seal. Both arms train on rlsf_train.jsonl and are scored on val; nothing here may
#    read data/splits/test.jsonl, and Week 3 opens it only after both arms are reported.
paths = [v for v in cfg['data'].values() if isinstance(v, str)]
assert not any('test' in Path(p).stem for p in paths), paths
assert cfg['data']['train_file'] == 'data/splits/rlsf_train.jsonl'
n_train = sum(1 for line in open(cfg['data']['train_file']) if line.strip())
print(f"training on {cfg['data']['train_file']}, {n_train} segments")
print(f"test split untouched: {sorted(Path(p).name for p in paths)}")

In [ ]:
# The rate is measured, not assumed: token counts from an existing usage artefact priced at
# whatever the client charges today. docs/budget.md quotes $2.36 for 500 steps.
from src.rlsf.pool import measured_per_call_usd

judge_client = make_judge_client(cfg)
rate = measured_per_call_usd(judge_client, cfg['judge']['model'])
per_rollout = rollout_batch(cfg)
calls = STEPS * per_rollout          # RL-Metric spends none of these
caps = cfg['rlsf']['caps']

print(f"RL-Metric   {STEPS} rollouts x {per_rollout} completions, 0 judge calls, $0.00")
print(f"RLSF-Judge  {STEPS} rollouts x {per_rollout} completions, {calls} judge calls at "
      f"concurrency {judge_concurrency(cfg)}")
print(f"            ${rate:.3e} per call -> ${calls * rate:.2f}  (docs/budget.md: $2.36)")
print(f"caps: {caps['max_steps']} steps, {caps['max_judge_calls']} calls, "
      f"${caps['max_judge_spend_usd']}")
assert STEPS <= caps['max_steps']
assert calls <= caps['max_judge_calls']
assert calls * rate <= caps['max_judge_spend_usd']

---
## 3 — Free wiring check, and the pace

In [ ]:
import time

SCRATCH = 'outputs/rlsf/train_wiring.jsonl'
started = time.perf_counter()
!{PY} manage.py rlsf_train --config {CONFIG} --cell w3_0.0 --skip_judge --steps 2 \
    --out {SCRATCH} --adapter_out models/rlsf_wiring --overwrite
elapsed = time.perf_counter() - started
print(f'\n{elapsed:.0f}s for 2 rollouts including model load')

In [ ]:
rows = [json.loads(x) for x in Path(SCRATCH).read_text().splitlines() if x.strip()]
assert len(rows) == 2, f'{len(rows)} rollouts logged, expected 2'
r = rows[0]
assert r['n_samples'] == per_rollout, (r['n_samples'], per_rollout)
assert r['n_groups'] == cfg['rlsf']['rollout']['prompts_per_step']
assert 'drift' in r, 'the step log carries no drift verdict'
assert set(r['raw']) == {'bleu', 'kiwi', 'judge'}
# Kiwi must vary: a constant means the COMET worker returned a placeholder, and the adequacy
# term would be silently absent from a reward that weights it at 0.7071.
assert r['raw']['kiwi'] != rows[1]['raw']['kiwi'], 'kiwi is constant across rollouts'
assert r['raw']['judge'] == 1.0, 'judge should be held flat under --skip_judge'

print(f"rollout 0: reward {r['reward_mean']:+.3f} sd {r['reward_sd']:.3f}, "
      f"{r['n_feasible']}/{r['n_samples']} feasible, "
      f"{r['degenerate_groups']}/{r['n_groups']} degenerate groups")
print(f"raw: " + '  '.join(f'{k} {v:.3f}' for k, v in r['raw'].items()))
print(f"marker_rate z {r['z']['marker_rate']:+.3f} +/- {r['z_se']['marker_rate']:.3f}")
print(f"drift: {r['drift']['reason']}")

In [ ]:
# Degenerate groups are the thing that makes a run train nothing: a flat group has zero
# advantage whatever the reward weights are.
flat = sum(x['degenerate_groups'] for x in rows) / sum(x['n_groups'] for x in rows)
print(f'{flat:.1%} of groups flat across the two rollouts')
if flat > 0.5:
    print('  more than half the groups carry no gradient; check temperature and G before '
          'spending a session on either arm')

In [ ]:
# Whether STEPS fits the session. The load is paid once, so the marginal rollout is what
# extrapolates; two rollouts is a coarse read of it, deliberately.
per_step = elapsed / 2
print(f'~{per_step:.0f}s per rollout (load included) -> {STEPS * per_step / 3600:.1f} h '
      f'for {STEPS} rollouts, per arm, before judge latency')
print('RLSF-Judge adds a serial judge block per rollout: docs/budget.md measured ~1.2 GPU-h '
      'of judge latency over 500 steps at concurrency 8.')
print()
print(f'If {STEPS} does not fit this session, lower STEPS in section 2 and rerun BOTH arms '
      f'at the new value. Record the change as a dated deviation in the pre-registration.')

---
## 4 — Arm 1: RL-Metric (`w3_0.0`)

ω = (0.7071, 0.7071, 0) at unit norm. `--skip_judge` because at ω₃ = 0 the judge cannot enter
the reward: this is the arm as declared, not a wiring check. **0 paid calls.**

The prediction: no rise in `marker_rate` z beyond the drift band, and no reason for
training-time Φ to move, since nothing optimizes it.

In [ ]:
!{PY} manage.py rlsf_train --config {CONFIG} --cell w3_0.0 --skip_judge --steps {STEPS}

---
## 5 — Read RL-Metric

In [ ]:
def read_arm(cell):
    """An arm's step log and the manifest written beside it."""
    log = arm_path(cfg['output']['step_log'], cell)
    rows = [json.loads(x) for x in log.read_text().splitlines() if x.strip()]
    man = json.loads(sidecar(log, 'manifest.json').read_text())
    return rows, man


def block_mean(rows, key, lo, hi):
    """Mean of a step-log field over rows[lo:hi], reaching into `raw` and `z` by dotted key."""
    head, _, tail = key.partition('.')
    values = [(r[head][tail] if tail else r[head]) for r in rows[lo:hi]]
    values = [v for v in values if v == v]
    return sum(values) / len(values) if values else float('nan')


def summarise(name, cell):
    rows, man = read_arm(cell)
    n = len(rows)
    opening = cfg['rlsf']['stop']['baseline_steps']
    print(f"{name} ({cell}): {n} rollouts, omega {man['omega']}")
    if man['outcome']['stop_reason']:
        print(f"  HALTED at step {man['outcome']['halted_at_step']}: "
              f"{man['outcome']['stop_reason']}")
    else:
        print(f"  ran to {n} rollouts without the drift rule firing")

    print(f"\n  {'quantity':22s} {'first ' + str(opening):>12s} {'last 5':>12s} {'delta':>10s}")
    for label, key in (
        ('reward mean', 'reward_mean'),
        ('reward sd', 'reward_sd'),
        ('BLEU', 'raw.bleu'),
        ('COMET-Kiwi', 'raw.kiwi'),
        ('Phi (judge)', 'raw.judge'),
        ('marker_rate z', 'z.marker_rate'),
        ('ttr z', 'z.ttr'),
        ('root_ttr z', 'z.root_ttr'),
        ('lex_density z', 'z.lex_density'),
        ('length ratio', 'length_ratio_mean'),
        ('degenerate frac', 'degenerate_frac'),
    ):
        a, b = block_mean(rows, key, 0, opening), block_mean(rows, key, n - 5, n)
        print(f"  {label:22s} {a:12.4f} {b:12.4f} {b - a:+10.4f}")
    # Reported because the pre-registration says it is, and read for its spread rather than
    # its level: group_z makes the combined reward a within-group z-score, mean ~0 by
    # construction. Whether the policy travelled is in the raw components.
    print('\n  reward mean sits near 0 by construction under group_z; read the raw rows')
    return rows, man


rows_metric, man_metric = summarise('RL-Metric', ARMS['RL-Metric'])

In [ ]:
# The drift verdict at every step, as the pre-registration says it is reported. The trailing
# rows are the ones that could have fired; the band is the run's own opening 20 steps.
def drift_tail(rows, k=10):
    testable = [r for r in rows if not r['drift']['reason'].endswith('are not complete')]
    print(f"{'step':>5s} {'z':>8s} {'baseline':>9s} {'delta':>8s} {'band':>8s}  tripped")
    for r in testable[-k:]:
        d = r['drift']
        print(f"{d['step']:5d} {r['z']['marker_rate']:+8.3f} {d['baseline']:+9.3f} "
              f"{d['delta']:+8.3f} {d['threshold']:8.3f}  {d['tripped']}")
    fired = [r for r in testable if r['drift']['tripped']]
    print(f"\n{len(testable)} testable steps, {len(fired)} tripped")
    return fired


fired_metric = drift_tail(rows_metric)

In [ ]:
# The pre-registration's asymmetry: RL-Metric is not predicted to trip. A trip here is
# reported as evidence against attributing drift to the judge term BEFORE it is called a
# 1.3% false alarm.
if fired_metric:
    print('RL-Metric tripped the drift rule.')
    print('Report this as evidence against attributing register drift to the judge term.')
    print('The 1.3% false-alarm rate from manage.py drift_oc is the second reading, not the '
          'first. Do not restart the arm.')
else:
    print('RL-Metric did not trip: the asymmetry the pre-registration predicts is intact '
          'so far, and the judge arm can be read against it.')

log = arm_path(cfg['output']['step_log'], ARMS['RL-Metric'])
assert not sidecar(log, 'usage.json').exists(), 'RL-Metric wrote a judge usage record'
print('\n0 paid calls, $0.00 spent on this arm')

---
## 6 — Gate before spending

RLSF-Judge buys 32,000 verdicts. Run this cell and read it before section 7: anything wrong
with RL-Metric means fix the code and rerun **both** arms, because the comparison is only
between arms trained on the same code.

In [ ]:
import statistics

opening_steps = cfg['rlsf']['stop']['baseline_steps']
checks = {
    'the arm ran': len(rows_metric) > 0,
    'every rollout is a full batch': all(r['n_samples'] == per_rollout for r in rows_metric),
    'kiwi varied across steps': len({round(r['raw']['kiwi'], 6) for r in rows_metric}) > 1,
    'bleu varied across steps': len({round(r['raw']['bleu'], 6) for r in rows_metric}) > 1,
    'judge stayed flat': {r['raw']['judge'] for r in rows_metric} == {1.0},
    'most groups carried a gradient':
        block_mean(rows_metric, 'degenerate_frac', 0, len(rows_metric)) < 0.5,
    # Not reward_mean: under group_z the combined reward is a within-group z-score, so its
    # mean is ~0 whatever the policy does. The raw components are what travel.
    'the policy travelled': any(
        abs(block_mean(rows_metric, f'raw.{c}', len(rows_metric) - 5, len(rows_metric))
            - block_mean(rows_metric, f'raw.{c}', 0, opening_steps))
        > statistics.pstdev([r['raw'][c] for r in rows_metric])
        for c in ('bleu', 'kiwi')),
    'no step lost samples to an unmeasured component':
        all(r['n_unmeasured'] == 0 for r in rows_metric),
}
for label, ok in checks.items():
    print(f"{'ok  ' if ok else 'FAIL'} {label}")

if not all(checks.values()):
    print('\nDo not run section 7. Fix the code, then rerun both arms from section 4.')
else:
    print('\nRL-Metric is clean. Section 7 may spend.')

---
## 7 — Arm 2: RLSF-Judge (`w3_2.0`)

In [ ]:
# Priced once more against the caps immediately before the spend (budget rule 1).
print(f"about to spend {calls} judge calls, ~${calls * rate:.2f}, against a "
      f"${caps['max_judge_spend_usd']} cap")
print(f"an early halt spends less; the cap is checked before each block of {per_rollout}")

In [ ]:
!{PY} manage.py rlsf_train --config {CONFIG} --cell w3_2.0 --steps {STEPS} --yes

---
## 8 — Read RLSF-Judge

In [ ]:
rows_judge, man_judge = summarise('RLSF-Judge', ARMS['RLSF-Judge'])
fired_judge = drift_tail(rows_judge)

In [ ]:
# What was actually spent, from the provider's own counts.
u = json.loads(sidecar(arm_path(cfg['output']['step_log'], ARMS['RLSF-Judge']),
                       'usage.json').read_text())
print(f"{u['calls']} calls, {u['prompt_tokens'] / u['calls']:.0f} in / "
      f"{u['completion_tokens'] / u['calls']:.0f} out per call")
print(f"${u['cost_usd']:.4f} total, ${u['per_call_usd']:.6f}/call  (planned ${calls * rate:.2f})")
print(f"judge blocks {u['wall_s'] / 60:.1f} min wall at concurrency {judge_concurrency(cfg)}, "
      f"{u['achieved_parallelism']:.1f}x achieved")

In [ ]:
# Unparseable verdicts. The pre-registration commits to measuring this rather than assuming
# it is zero: an unparseable verdict marks a sample infeasible here and, with the judge held
# flat, cannot in RL-Metric. That asymmetry is exactly this number.
lost = sum(r['n_unmeasured'] for r in rows_judge)
total = sum(r['n_samples'] for r in rows_judge)
print(f'{lost}/{total} samples unmeasured ({lost / total:.2%}); those are floored, not scored 1')
print(f'RL-Metric lost {sum(r["n_unmeasured"] for r in rows_metric)} to the same cause')

---
## 9 — The four figures the predictions are scored against

Three of them are in the step logs. The fourth — held-out register distance over `ttr`,
`root_ttr`, `marker_rate` — is a val-set measurement of the trained adapters and is not
computed here. It belongs to the evaluation pass, with the val row of the results table.

In [ ]:
n_j, n_m = len(rows_judge), len(rows_metric)
opening = cfg['rlsf']['stop']['baseline_steps']

print(f"{'':22s} {'RL-Metric':>22s} {'RLSF-Judge':>22s}")
for label, key in (('Phi (judge)', 'raw.judge'),
                   ('marker_rate z', 'z.marker_rate'),
                   ('COMET-Kiwi', 'raw.kiwi'),
                   ('BLEU', 'raw.bleu')):
    dm = (block_mean(rows_metric, key, n_m - 5, n_m)
          - block_mean(rows_metric, key, 0, opening))
    dj = (block_mean(rows_judge, key, n_j - 5, n_j)
          - block_mean(rows_judge, key, 0, opening))
    print(f"{label:22s} {dm:+22.4f} {dj:+22.4f}")
print(f"\nlast 5 steps minus the opening {opening}, per arm, against its own initialization")

In [ ]:
# Prediction 3 asks whether Kiwi is flat against the run's OWN step-to-step spread, not
# against zero. The step means are a noisy series; the comparison is a movement against it.
import statistics


def kiwi_verdict(rows, name):
    series = [r['raw']['kiwi'] for r in rows]
    spread = statistics.pstdev(series)
    move = (sum(series[-5:]) / 5) - (sum(series[:opening]) / opening)
    flat = abs(move) <= spread
    print(f"{name:12s} kiwi moved {move:+.4f} against a step-to-step sd of {spread:.4f} "
          f"-> {'flat' if flat else 'NOT flat'}")
    return flat


kiwi_verdict(rows_metric, 'RL-Metric')
kiwi_verdict(rows_judge, 'RLSF-Judge')

In [ ]:
# The pairing is the claim, not either series alone: Phi and marker_rate z predicted to move
# together over the same steps.
def pearson(a, b):
    ma, mb = sum(a) / len(a), sum(b) / len(b)
    da, db = [x - ma for x in a], [y - mb for y in b]
    den = math.sqrt(sum(x * x for x in da) * sum(y * y for y in db))
    return sum(x * y for x, y in zip(da, db)) / den if den else float('nan')


phi = [r['raw']['judge'] for r in rows_judge]
mrz = [r['z']['marker_rate'] for r in rows_judge]
print(f'RLSF-Judge: corr(Phi, marker_rate z) over {len(phi)} steps = {pearson(phi, mrz):+.3f}')
print(f'  Phi   {phi[0]:.3f} -> {phi[-1]:.3f}')
print(f'  z     {mrz[0]:+.3f} -> {mrz[-1]:+.3f}')
print('\nStep means are autocorrelated, so this correlation describes the trajectory; it is '
      'not a test.')

In [ ]:
# The halt, stated plainly, because it is the result the pre-registration asks to be reported.
for name, rows, fired in (('RL-Metric', rows_metric, fired_metric),
                          ('RLSF-Judge', rows_judge, fired_judge)):
    man = read_arm(ARMS[name])[1]
    step = man['outcome']['halted_at_step']
    if step is None:
        print(f'{name:12s} ran {len(rows)} rollouts, drift rule did not fire')
    else:
        print(f'{name:12s} HALTED at step {step} of {STEPS}: {man["outcome"]["stop_reason"]}')
print('\nA halt is reported at its step. Neither arm is restarted under a loosened rule.')

---
## 10 — Take the artefacts off the host

Both step logs, both manifests, the judge usage record and both adapters. The adapters are
what the val evaluation loads; without them the arms have to be retrained.

In [ ]:
artefacts = []
for cell in ARMS.values():
    log = arm_path(cfg['output']['step_log'], cell)
    artefacts += [log, sidecar(log, 'manifest.json')]
    if sidecar(log, 'usage.json').exists():
        artefacts.append(sidecar(log, 'usage.json'))
    artefacts.append(arm_path(cfg['output']['adapter_dir'], cell))

for p in artefacts:
    size = (sum(f.stat().st_size for f in p.rglob('*') if f.is_file())
            if p.is_dir() else p.stat().st_size)
    print(f'{size / 1e6:8.2f} MB  {p}{"/" if p.is_dir() else ""}')
    assert p.exists(), p

In [ ]:
import shutil

from google.colab import drive

drive.mount('/content/drive')
dest = Path('/content/drive/MyDrive/Style-Aware-MT/rlsf')
dest.mkdir(parents=True, exist_ok=True)
for p in artefacts:
    target = dest / p.name
    if p.is_dir():
        shutil.copytree(p, target, dirs_exist_ok=True)
    else:
        shutil.copy2(p, target)
    print(f'copied {p} -> {target}')

In [ ]:
# The commit the run leaves behind: logs and manifests into the repo, adapters out of band.
print('git add ' + ' '.join(str(p) for p in artefacts if not p.is_dir()))
print('\nAdapters are too large for the repo; they stay in Drive and the manifest names them:')
for cell in ARMS.values():
    print(f"  {read_arm(cell)[1]['outcome']['adapter_dir']}")